# CivicWater Sentinel — P2 · Step 1: Hour-of-week baseline

The first step of the model ladder (design doc §3, Table 2). For every pressure sensor, the
expected reading is the **median of 2018 for the same hour of the week**. It is deliberately
simple: it is the floor that XGBoost (step 2) and the GRU (step 3) must beat, and it already
gives the detector real, non-mock expectations.

| | |
|---|---|
| **Input** | `data/v1/scada_2018.csv.gz` (P1). 2019 is never read here. |
| **Output** | `models/baseline-hourofweek@0.1.0.json` — profile + σ per sensor |
| **Runtime** | `sentinel.forecasting.baseline.HourOfWeekForecaster` (plain Python) |
| **Runs on** | anywhere; the profile is tiny and is scored on LinuxONE |

In [ ]:
# --- Setup (same for every P2 notebook) --------------------------------------------------
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")

# Repository root = first parent folder that contains src/sentinel (works on the VM and in Colab)
REPO = Path(os.environ.get("REPO") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sentinel").exists()))
sys.path[:0] = [str(REPO / "src"), str(REPO / "notebooks" / "p2_forecast")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import p2lib
FAST = os.environ.get("P2_FAST") == "1"   # smoke-test mode: smaller training, same code path
print("repo   :", REPO)
print("data   :", p2lib.data_dir())
print("models :", p2lib.models_dir())
print("on LinuxONE (s390x):", p2lib.on_linuxone(), "| FAST mode:", FAST)

## Load 2018 SCADA data

Same pattern as the organisers' *Client Retention* demo: CSV → pandas DataFrame. Sensor IDs and
their order come from P1's `sensors.csv`, read with the backend's own registry parser, so the
notebook and the API agree on every ID.

In [ ]:
layout = p2lib.load_layout()
df = p2lib.load_scada(2018)
P = list(layout.pressure)
print(df.shape, "|", df.index.min(), "→", df.index.max())
print(f"{len(layout.pressure)} pressure, {len(layout.flow)} flow, {len(layout.tank_level)} level sensors")
print("missing values in pressure columns:", int(df[P].isna().sum().sum()))
df[P[:4] + list(layout.flow) + list(layout.tank_level)].describe().T.round(2)

In [ ]:
# Daily pattern of three sensors during one week: the shape the baseline learns
week = df.loc["2018-03-05":"2018-03-11", P[:3]]
ax = week.plot(figsize=(14, 4), title="Pressure, one week of March 2018 (m)")
ax.set_ylabel("m"); plt.show()

## Leak-aware training

2018 is not a clean year: the leak report lists leaks that ran for weeks (design doc §2). Every
window covered by a reported leak is excluded from fitting, so the "normal" profile is not
learned from a leaking network.

In [ ]:
leaks = p2lib.load_leaks_2018()
free = p2lib.leak_free_mask(df.index, leaks)
print(f"{len(leaks)} leaks in the 2018 report | leak-free windows: {free.mean():.1%}")
leaks[["pipe", "start", "end", "leak_size"]].head(14)

## Fit on Jan–Oct, validate on Nov–Dec

Validation stays inside 2018, so 2019 remains an untouched test year. σ per sensor is the robust
spread (1.4826·MAD) of the **validation** residuals: it is what P4 divides by in
`z = (expected − observed) / σ`.

In [ ]:
train, val = p2lib.split(df)
prof_tr = p2lib.fit_hour_of_week(train[free.loc[train.index]], layout)
resid = val[P] - p2lib.predict_hour_of_week(prof_tr, val)

mae = p2lib.mae_by_sensor(resid)
sigma = p2lib.robust_sigma(resid)
contrast = p2lib.leak_contrast(resid, sigma, free)
print(f"validation MAE (mean of sensors): {mae.mean():.3f} m | worst sensor: {mae.idxmax()} {mae.max():.3f} m")
print("leak contrast on validation (|z| inside / outside leaks):", None if contrast is None else round(contrast, 2))
mae.sort_values(ascending=False).head(8).round(3)

## Export the model (fitted on all leak-free 2018 windows)

In [ ]:
VERSION = "0.1.0"
prof = p2lib.fit_hour_of_week(df[free], layout)
path = p2lib.models_dir() / f"baseline-hourofweek@{VERSION}.json"
meta = p2lib.artefact_meta(
    "baseline-hourofweek", VERSION, layout,
    trained_on="2018, leak-free windows, median per hour of week",
    validation={"period": "2018-11-01..2018-12-31", "mae_m_mean": round(float(mae.mean()), 4),
                "mae_m": mae.round(4).to_dict(), "leak_contrast": contrast},
    sigma_m=sigma.round(4).to_dict(),
    profile={s: [round(float(v), 4) for v in prof[s]] for s in P},
)
p2lib.write_json(path, meta)
print("saved", path, f"({path.stat().st_size/1024:.0f} KB)")

## Runtime check: the API forecaster gives the same numbers

The backend never sees pandas: it loads the JSON into `HourOfWeekForecaster` and receives
`ForecastRequest` objects. Here we build those requests from real rows and compare.

In [ ]:
from sentinel.forecasting.registry import load_forecaster
model = load_forecaster("baseline", path)
print(model.info)

expected = p2lib.predict_hour_of_week(prof, val)
stamps = p2lib.complete_rows(val, layout).sample(50, random_state=1).index
diff = max(abs(model.predict(p2lib.build_request(df, ts, layout, history=1)).expected_pressures[s]
               - expected.loc[ts, s]) for ts in stamps for s in P)
print(f"max |runtime − notebook| over 50 windows × {len(P)} sensors: {diff:.2e} m")
assert diff < 1e-3

## Latency of one forecast (one 5-minute window, all sensors)

In [ ]:
req = p2lib.build_request(df, stamps[0], layout, history=1)
lat = p2lib.benchmark(lambda: model.predict(req), repeats=200)
print({k: round(v, 4) for k, v in lat.items()}, "| machine:", p2lib.accelerator_report()["machine"])
meta["latency_ms"] = {p2lib.accelerator_report()["machine"]: lat}
p2lib.write_json(path, meta);

## Summary

* The baseline is the **floor** of the ladder: its validation MAE is the number to beat.
* `sigma_m` in the JSON is P4's σ until a better model replaces the baseline.
* Next: `02_xgboost_snapml.ipynb`.